# Mem0: user-scoped semantic memory

Mem0 extracts useful facts from conversations and makes them searchable later. This experiment adds facts for one user, searches them, then demonstrates user isolation. The provider-backed example requires an LLM and embedding provider; keep keys in environment variables, never in the notebook.

## Which memory type should I use?

Choose based on what must be remembered and what question the system must answer:

| Need | Memory type to choose | Typical fit |
|---|---|---|
| Continue the same conversation or workflow | Thread/checkpoint state | Recent messages, pending step, tool result |
| Personalize future conversations | Semantic/profile memory | Preferences, stable facts, goals |
| Recall a particular past event | Episodic memory | Decisions, meetings, dated interactions |
| Repeat a validated workflow | Procedural/experience memory | How-to steps, successful task lessons |
| Answer from reference documents | Document/RAG memory | Policies, manuals, knowledge bases |
| Connect people/things across facts | Entity/relationship graph | Multi-hop questions and linked entities |
| Answer what changed and when | Temporal memory | Versioned facts, evolving relationships |
| Keep large histories manageable | Hierarchical/consolidated memory | Profile summary plus recent episodes/archive |

Do not pick the most complex tool by default. Choose the simplest type that meets measured needs; many systems combine two types (for example, thread checkpoints plus a separate profile store). Retrieval must always be scoped to the authenticated user/tenant before results are sent to an LLM.

## Why this experiment uses this tool

**Primary type here:** Semantic / user-profile memory.

**When to use it:** Personalization across separate chats: preferences, stable facts, and goals that should be selectively retrieved for one user.

**Choose this tool when:** Choose Mem0 when you want an add/search/update-oriented semantic memory layer and can configure its LLM, embeddings, and storage. Use a thread checkpointer for same-chat continuity; use a RAG index for authoritative documents; use a temporal graph when relationship history is central.

**About this notebook:** This notebook uses Mem0 for extraction and retrieval, then an LLM answers using retrieved memories. It is the collection?s provider-backed memory example.


## Setup

Install `mem0ai` in the notebook environment. Set `OPENAI_API_KEY` in the environment before running the provider-backed experiment. The default local vector store uses a folder next to this notebook.

In [ ]:
%pip install -q mem0ai openai

In [ ]:
import os
from mem0 import Memory

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this cell.")

memory = Memory.from_config({
    "llm": {
        "provider": "openai",
        "config": {"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]},
    },
    "embedder": {
        "provider": "openai",
        "config": {"model": "text-embedding-3-small", "api_key": os.environ["OPENAI_API_KEY"]},
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {"path": "./mem0_data", "embedding_model_dims": 1536},
    },
})

memory.add(
    "I prefer vegetarian food and I am training for a half marathon.",
    user_id="demo-user",
)
print(memory.search("What food do I prefer?", user_id="demo-user"))
print(memory.search("What food does the other user prefer?", user_id="another-user"))

## What to observe

The first search should retrieve relevant information for `demo-user`; the second is scoped to a different user and should not expose that user's data. Repeat with a correction or changed preference and inspect how the memory is updated. This uses a real provider and may incur charges. Treat user IDs as authorization boundaries in production and configure persistence/retention intentionally.

## Ten Mem0 + LLM examples

These examples use the configured Mem0 instance, not a local token-overlap mock. `memory.add(...)` uses the configured Mem0 LLM to extract/update memories, `memory.search(...)` retrieves relevant memories, and the OpenAI chat model below answers using only those retrieved memories. Run setup cells above first, then run examples individually. Each example uses a separate user ID to avoid mixing data. Provider-backed calls can incur charges; add/search/answer calls use your configured services.

In [ ]:
from openai import OpenAI

answer_llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def mem0_ask(query, user_id):
    """Retrieve Mem0 memories, then ask an LLM to answer from retrieved evidence."""
    result = memory.search(query, user_id=user_id)
    hits = result.get("results", []) if isinstance(result, dict) else result
    memories = [hit.get("memory", "") for hit in hits if hit.get("memory")]
    context = "\n".join(f"- {item}" for item in memories) or "No relevant memories were retrieved."
    response = answer_llm.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": "Answer using only the supplied memory context. If it does not contain the answer, say: I don't have that information in memory."},
            {"role": "user", "content": f"Memory context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    return {"memories": memories, "answer": response.choices[0].message.content}

print("Mem0 retrieval + LLM answer helper ready.")

### Example 1: Profile recall

In [ ]:
user_id = 'mem0-demo-profile'
memory.add('I am a data analyst and I use Python to create weekly reports.', user_id=user_id)
print(mem0_ask('What is my role and what do I use Python for?', user_id))

### Example 2: Preference recall

In [ ]:
user_id = 'mem0-demo-preferences'
memory.add('I prefer vegetarian meals and avoid dairy products.', user_id=user_id)
print(mem0_ask('What food preferences should you remember for me?', user_id))

### Example 3: Goal recall

In [ ]:
user_id = 'mem0-demo-goal'
memory.add('I am training for a half marathon in October and run three times a week.', user_id=user_id)
print(mem0_ask('What fitness goal am I working toward?', user_id))

### Example 4: Episodic memory

In [ ]:
user_id = 'mem0-demo-episode'
memory.add('On 2026-04-12 I visited Kyoto and especially enjoyed the botanical garden.', user_id=user_id)
print(mem0_ask('Which place did I enjoy visiting in Kyoto?', user_id))

### Example 5: Update a preference

In [ ]:
user_id = 'mem0-demo-update'
memory.add('My preferred code example language is JavaScript.', user_id=user_id)
memory.add('My preference changed: use TypeScript instead of JavaScript for code examples.', user_id=user_id)
print(mem0_ask('What programming language do I currently prefer for code examples?', user_id))

### Example 6: User isolation

In [ ]:
user_id = 'mem0-demo-isolation-a'
memory.add('I prefer concise answers with a short summary first.', user_id=user_id)
print("Search under the different user ID (should not retrieve user A?s fact):")
print(mem0_ask('What answer style does the user prefer?', 'mem0-demo-isolation-b'))

### Example 7: Temporal context

In [ ]:
user_id = 'mem0-demo-temporal'
memory.add('On 2026-02-01 our project used SQLite. On 2026-06-01 we migrated to PostgreSQL.', user_id=user_id)
print(mem0_ask('Which database did the project migrate to in June?', user_id))

### Example 8: Procedural memory

In [ ]:
user_id = 'mem0-demo-procedure'
memory.add('When explaining code to me, start with a short summary and then show a runnable example.', user_id=user_id)
print(mem0_ask('How should a code explanation be structured for me?', user_id))

### Example 9: Related facts

In [ ]:
user_id = 'mem0-demo-related'
memory.add('Mira works at Northstar Labs. Northstar Labs builds the Aurora assistant for researchers.', user_id=user_id)
print(mem0_ask('Which assistant does Mira?s company build?', user_id))

### Example 10: Grounded RAG memory

In [ ]:
user_id = 'mem0-demo-rag'
memory.add('Handbook section 4 says refund requests must be submitted within 30 days of purchase.', user_id=user_id)
print(mem0_ask('According to the handbook, how long is the refund window? Include the source section if retrieved.', user_id))